## Install the required libraries

In [1]:
! pip install ollama


  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)

  Attempting uninstall: httpx

    Found existing installation: httpx 0.25.2

    Uninstalling httpx-0.25.2:

      Successfully uninstalled httpx-0.25.2

   ---------------------------------------- 0/2 [httpx]
   ---------------------------------------- 0/2 [httpx]
   ---------------------------------------- 0/2 [httpx]
   ---------------------------------------- 0/2 [httpx]
   -------------------- ------------------- 1/2 [ollama]
   ---------------------------------------- 2/2 [ollama]



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.56.0 requires anyio<5.0.0,>=4.8.0, but you have anyio 3.7.1 which is incompatible.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\Suryaprakash\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Define the Language model and Embedding model

In [3]:
import ollama

language_model = "hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF:latest"
embedding_model = "hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF:latest"


## Loading the Data

In [5]:
datasets = []
with open("cat-facts.txt", "r", encoding="utf-8", errors="replace") as f:
    datasets = f.readlines()
print(f"Loaded {len(datasets)} cat facts")

Loaded 150 cat facts


In [6]:
VECTOR_DB = []

## Defining Embedding function

In [11]:
def add_chunks_to_vector_db(chunk):
    embeddings = ollama.embed(embedding_model, input=chunk)['embeddings'][0]
    VECTOR_DB.append((chunk, embeddings))

## Vectorization and Storing in VectorDB

In [13]:
for i, chunk in enumerate(datasets):
    add_chunks_to_vector_db(chunk)
    print(f"Added chunk {i+1}/{len(datasets)} to vector database")

Added chunk 1/150 to vector database
Added chunk 2/150 to vector database
Added chunk 3/150 to vector database
Added chunk 4/150 to vector database
Added chunk 5/150 to vector database
Added chunk 6/150 to vector database
Added chunk 7/150 to vector database
Added chunk 8/150 to vector database
Added chunk 9/150 to vector database
Added chunk 10/150 to vector database
Added chunk 11/150 to vector database
Added chunk 12/150 to vector database
Added chunk 13/150 to vector database
Added chunk 14/150 to vector database
Added chunk 15/150 to vector database
Added chunk 16/150 to vector database
Added chunk 17/150 to vector database
Added chunk 18/150 to vector database
Added chunk 19/150 to vector database
Added chunk 20/150 to vector database
Added chunk 21/150 to vector database
Added chunk 22/150 to vector database
Added chunk 23/150 to vector database
Added chunk 24/150 to vector database
Added chunk 25/150 to vector database
Added chunk 26/150 to vector database
Added chunk 27/150 to

In [14]:
def cosine_similarity(vec1, vec2):
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    magnitude1 = sum(a ** 2 for a in vec1) ** 0.5
    magnitude2 = sum(b ** 2 for b in vec2) ** 0.5
    if magnitude1 == 0 or magnitude2 == 0:
        return 0.0
    return dot_product / (magnitude1 * magnitude2)

## Retrieval pipeline

In [15]:
def retrieval(query_embedding, top_k=5):
    similarities = []
    for chunk, embedding in VECTOR_DB:
        sim = cosine_similarity(query_embedding, embedding)
        similarities.append((chunk, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

In [16]:
query = "What are some interesting facts about cats?"
query_embedding = ollama.embed(embedding_model, input=query)['embeddings'][0]

In [23]:
result = retrieval(query_embedding, top_k=5)
print("Top 5 retrieved chunks:\n")
for i, (chunk, sim) in enumerate(result):
    print(f"{i+1}. Similarity: {sim:.4f}\n{chunk}\n")

Top 5 retrieved chunks:

1. Similarity: 0.4201
The chlorine in fresh tap water irritates sensitive parts of the cat’s nose. Let tap water sit for 24 hours before giving it to a cat.

2. Similarity: 0.3395
Cats spend nearly 1/3 of their waking hours cleaning themselves.


3. Similarity: 0.3002
Cats sleep 16 to 18 hours per day. When cats are asleep, they are still alert to incoming stimuli. If you poke the tail of a sleeping cat, it will respond accordingly.


4. Similarity: 0.2994
Cats make about 100 different sounds. Dogs make only about 10.


5. Similarity: 0.2994
Cats make about 100 different sounds. Dogs make only about 10.


